In [22]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path

# =========================================================
# CONFIGURACIÓN
# =========================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Rutas
ruta_carpeta_aire = "../data/raw/aire_miteco"
ruta_salida = "../data/processed"

os.makedirs(ruta_salida, exist_ok=True)

# =========================================================
# FUNCIONES
# =========================================================

def leer_csv_robusto(ruta):
    for sep in [";", ",", "\t"]:
        for encoding in ["utf-8", "latin1", "cp1252"]:
            try:
                df = pd.read_csv(ruta, sep=sep, encoding=encoding, low_memory=False)
                if df.shape[1] > 1:
                    print(f"OK -> {Path(ruta).name} | sep='{sep}' | encoding='{encoding}' | shape={df.shape}")
                    return df
            except Exception:
                pass
    raise ValueError(f"No se pudo leer el archivo: {ruta}")

def obtener_estacion_anio(mes):
    if mes in [12, 1, 2]:
        return "Invierno"
    elif mes in [3, 4, 5]:
        return "Primavera"
    elif mes in [6, 7, 8]:
        return "Verano"
    return "Otoño"

def obtener_franja(hora):
    hora_real = hora - 1  # h01 = 00:00
    if 0 <= hora_real <= 5:
        return "Madrugada"
    elif 6 <= hora_real <= 11:
        return "Mañana"
    elif 12 <= hora_real <= 17:
        return "Tarde"
    return "Noche"

# =========================================================
# 1) CARGAR Y UNIR LOS CSV
# =========================================================

archivos_aire = glob.glob(os.path.join(ruta_carpeta_aire, "*.csv"))

print(f"Archivos encontrados: {len(archivos_aire)}")
for archivo in archivos_aire:
    print("-", Path(archivo).name)

if len(archivos_aire) == 0:
    raise ValueError(f"No se encontraron archivos CSV en: {os.path.abspath(ruta_carpeta_aire)}")

lista_dfs = []

for archivo in archivos_aire:
    df_temp = leer_csv_robusto(archivo)
    df_temp["archivo_origen"] = Path(archivo).name
    lista_dfs.append(df_temp)

df_aire = pd.concat(lista_dfs, ignore_index=True)

print("\nShape unido:", df_aire.shape)
display(df_aire.head())

# =========================================================
# 2) NORMALIZAR COLUMNAS
# =========================================================

df_aire.columns = (
    df_aire.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

print("\nColumnas:")
print(df_aire.columns.tolist())

# =========================================================
# 3) ASEGURAR TIPOS NUMÉRICOS EN COLUMNAS CLAVE
# =========================================================

columnas_clave = ["provincia", "municipio", "estacion", "magnitud", "anno", "mes", "dia"]
for col in columnas_clave:
    df_aire[col] = pd.to_numeric(df_aire[col], errors="coerce")

# =========================================================
# 4) FILTRAR ZARAGOZA
# =========================================================

# Provincia Zaragoza = 50
df_zgz_prov = df_aire[df_aire["provincia"] == 50].copy()

print("\nShape provincia Zaragoza:", df_zgz_prov.shape)

# Resumen para confirmar municipio
resumen_municipios = (
    df_zgz_prov.groupby("municipio")
    .agg(
        n_filas=("municipio", "size"),
        n_estaciones=("estacion", "nunique"),
        n_magnitudes=("magnitud", "nunique")
    )
    .sort_values(["n_estaciones", "n_filas"], ascending=False)
    .reset_index()
)

print("\nResumen municipios provincia Zaragoza:")
display(resumen_municipios.head(20))

# Zaragoza ciudad = 297 según tus datos
df_zaragoza = df_zgz_prov[df_zgz_prov["municipio"] == 297].copy()

print("\nShape Zaragoza ciudad:", df_zaragoza.shape)
print("Estaciones únicas:", sorted(df_zaragoza["estacion"].dropna().unique().tolist()))
print("Número de estaciones:", df_zaragoza["estacion"].nunique())
print("Magnitudes únicas:", sorted(df_zaragoza["magnitud"].dropna().unique().tolist()))
print("Número de magnitudes:", df_zaragoza["magnitud"].nunique())

display(df_zaragoza.head())

# =========================================================
# 5) LIMPIAR COLUMNAS HORARIAS
# =========================================================

columnas_horas = [f"h{i:02d}" for i in range(1, 25)]

for col in columnas_horas:
    df_zaragoza[col] = (
        df_zaragoza[col]
        .astype(str)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
        .str.replace(",", ".", regex=False)
    )
    df_zaragoza[col] = pd.to_numeric(df_zaragoza[col], errors="coerce")

print("\nNulos por columnas horarias:")
display(df_zaragoza[columnas_horas].isnull().sum().sort_values(ascending=False))

# =========================================================
# 6) PASAR DE FORMATO ANCHO A LARGO
# =========================================================

id_vars = [
    "provincia",
    "municipio",
    "estacion",
    "magnitud",
    "punto_muestreo",
    "anno",
    "mes",
    "dia",
    "archivo_origen"
]

df_zaragoza_long = df_zaragoza.melt(
    id_vars=id_vars,
    value_vars=columnas_horas,
    var_name="hora",
    value_name="valor"
)

print("\nShape formato largo:", df_zaragoza_long.shape)
display(df_zaragoza_long.head())

# =========================================================
# 7) CREAR FECHA Y DATETIME
# =========================================================

df_zaragoza_long["hora"] = (
    df_zaragoza_long["hora"]
    .str.replace("h", "", regex=False)
    .astype(int)
)

df_zaragoza_long["fecha"] = pd.to_datetime(
    dict(
        year=df_zaragoza_long["anno"],
        month=df_zaragoza_long["mes"],
        day=df_zaragoza_long["dia"]
    ),
    errors="coerce"
)

# h01 = 00:00, h24 = 23:00
df_zaragoza_long["datetime"] = (
    df_zaragoza_long["fecha"] +
    pd.to_timedelta(df_zaragoza_long["hora"] - 1, unit="h")
)

# =========================================================
# 8) VARIABLES TEMPORALES
# =========================================================

df_zaragoza_long["anio"] = df_zaragoza_long["datetime"].dt.year
df_zaragoza_long["mes_num"] = df_zaragoza_long["datetime"].dt.month
df_zaragoza_long["dia_mes"] = df_zaragoza_long["datetime"].dt.day
df_zaragoza_long["dia_semana_num"] = df_zaragoza_long["datetime"].dt.dayofweek
df_zaragoza_long["dia_semana"] = df_zaragoza_long["datetime"].dt.day_name()
df_zaragoza_long["fin_de_semana"] = df_zaragoza_long["dia_semana_num"].isin([5, 6])

mapa_meses = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

mapa_dias = {
    "Monday": "Lunes",
    "Tuesday": "Martes",
    "Wednesday": "Miércoles",
    "Thursday": "Jueves",
    "Friday": "Viernes",
    "Saturday": "Sábado",
    "Sunday": "Domingo"
}

df_zaragoza_long["mes_nombre"] = df_zaragoza_long["mes_num"].map(mapa_meses)
df_zaragoza_long["dia_semana"] = df_zaragoza_long["dia_semana"].map(mapa_dias)
df_zaragoza_long["estacion_anio"] = df_zaragoza_long["mes_num"].apply(obtener_estacion_anio)
df_zaragoza_long["franja_horaria"] = df_zaragoza_long["hora"].apply(obtener_franja)

# =========================================================
# 9) MAPEAR MAGNITUDES A CONTAMINANTES
# =========================================================

mapa_magnitudes = {
    1: "SO2",
    6: "CO",
    7: "NO",
    8: "NO2",
    9: "PM2.5",
    10: "PM10",
    12: "NOx",
    14: "O3",
    20: "Tolueno",
    30: "Benceno",
    35: "Etilbenceno",
    37: "m+p-Xileno",
    38: "o-Xileno",
    42: "Hidrocarburos totales",
    43: "Metano",
    44: "Hidrocarburos no metánicos"
}

df_zaragoza_long["contaminante"] = df_zaragoza_long["magnitud"].map(mapa_magnitudes)
df_zaragoza_long["contaminante"] = df_zaragoza_long.apply(
    lambda x: f"Magnitud_{int(x['magnitud'])}" if pd.isna(x["contaminante"]) else x["contaminante"],
    axis=1
)

# =========================================================
# 10) REORDENAR COLUMNAS
# =========================================================

columnas_finales = [
    "provincia",
    "municipio",
    "estacion",
    "magnitud",
    "contaminante",
    "punto_muestreo",
    "archivo_origen",
    "anno",
    "mes",
    "dia",
    "hora",
    "valor",
    "fecha",
    "datetime",
    "anio",
    "mes_num",
    "mes_nombre",
    "dia_mes",
    "dia_semana_num",
    "dia_semana",
    "fin_de_semana",
    "estacion_anio",
    "franja_horaria"
]

df_zaragoza_long = df_zaragoza_long[columnas_finales].copy()

# =========================================================
# 11) COMPROBACIONES FINALES
# =========================================================

print("\nShape final aire Zaragoza:", df_zaragoza_long.shape)
print("Nulos en valor:", df_zaragoza_long["valor"].isnull().sum())
print("Nulos en fecha:", df_zaragoza_long["fecha"].isnull().sum())
print("Nulos en datetime:", df_zaragoza_long["datetime"].isnull().sum())

print("\nContaminantes detectados:")
print(sorted(df_zaragoza_long["contaminante"].dropna().unique().tolist()))

print("\nRango de fechas:")
print(df_zaragoza_long["datetime"].min(), "->", df_zaragoza_long["datetime"].max())

display(df_zaragoza_long.head())

# =========================================================
# 12) GUARDAR RESULTADOS
# =========================================================

df_aire.to_csv(f"{ruta_salida}/aire_total_unido.csv", index=False)
df_zaragoza.to_csv(f"{ruta_salida}/aire_zaragoza_ancho.csv", index=False)
df_zaragoza_long.to_csv(f"{ruta_salida}/aire_zaragoza_long.csv", index=False)

print("\nArchivos guardados correctamente:")
print(f"- {ruta_salida}/aire_total_unido.csv")
print(f"- {ruta_salida}/aire_zaragoza_ancho.csv")
print(f"- {ruta_salida}/aire_zaragoza_long.csv")

Archivos encontrados: 4
- aire_2020.csv
- aire_2021.csv
- aire_2022.csv
- aire_2023.csv
OK -> aire_2020.csv | sep=';' | encoding='utf-8' | shape=(818999, 32)
OK -> aire_2021.csv | sep=';' | encoding='utf-8' | shape=(865660, 32)
OK -> aire_2022.csv | sep=';' | encoding='utf-8' | shape=(859188, 32)
OK -> aire_2023.csv | sep=';' | encoding='utf-8' | shape=(1092135, 32)

Shape unido: (3635982, 33)


,provincia,municipio,estacion,magnitud,punto muestreo,anno,mes,dia,h01,h02,h03,h04,h05,h06,h07,h08,h09,h10,h11,h12,h13,h14,h15,h16,h17,h18,h19,h20,h21,h22,h23,h24,archivo_origen
0,43,162,5,8,43162005_8_8,2020,2,6,2,3,2,5,6,6,7,6,4,7,6,7,11,9,4,3,3,4,4,3,3,2,3,6,aire_2020.csv
1,43,171,1,1,43171001_1_38,2020,2,6,4,5,6,5,6,6,8,7,8,8,8,7,6,5,4,3,3,2,3,3,2,3,2,2,aire_2020.csv
2,43,171,1,8,43171001_8_8,2020,2,6,27,20,16,20,24,37,53,50,43,31,18,7,7,7,7,6,10,21,28,33,39,68,65,60,aire_2020.csv
3,43,162,2,8,43162002_8_8,2020,2,6,10,9,13,15,18,20,29,28,10,7,6,7,9,7,5,5,7,9,14,21,13,5,12,10,aire_2020.csv
4,43,171,1,14,43171001_14_6,2020,2,6,13,18,23,20,17,15,4,2,7,29,52,71,75,74,73,80,74,59,53,53,42,4,4,6,aire_2020.csv



Columnas:
['provincia', 'municipio', 'estacion', 'magnitud', 'punto_muestreo', 'anno', 'mes', 'dia', 'h01', 'h02', 'h03', 'h04', 'h05', 'h06', 'h07', 'h08', 'h09', 'h10', 'h11', 'h12', 'h13', 'h14', 'h15', 'h16', 'h17', 'h18', 'h19', 'h20', 'h21', 'h22', 'h23', 'h24', 'archivo_origen']

Shape provincia Zaragoza: (66996, 33)

Resumen municipios provincia Zaragoza:


,municipio,n_filas,n_estaciones,n_magnitudes
0,297,54369,8,7
1,8,6209,1,5
2,59,3287,1,3
3,67,1670,1,5
4,101,1461,1,1



Shape Zaragoza ciudad: (54369, 33)
Estaciones únicas: [26, 29, 32, 36, 37, 38, 39, 40]
Número de estaciones: 8
Magnitudes únicas: [1, 6, 7, 8, 9, 10, 14]
Número de magnitudes: 7


...


Shape formato largo: (1304856, 11)


,provincia,municipio,estacion,magnitud,punto_muestreo,anno,mes,dia,archivo_origen,hora,valor
0,50,297,29,10,50297029_10_49,2020,2,6,aire_2020.csv,h01,19.48
1,50,297,29,14,50297029_14_6,2020,2,6,aire_2020.csv,h01,14.61
2,50,297,32,1,50297032_1_38,2020,2,6,aire_2020.csv,h01,1.77
3,50,297,32,6,50297032_6_48,2020,2,6,aire_2020.csv,h01,0.22
4,50,297,32,8,50297032_8_8,2020,2,6,aire_2020.csv,h01,33.43



Shape final aire Zaragoza: (1304856, 23)
Nulos en valor: 44840
Nulos en fecha: 0
Nulos en datetime: 0

Contaminantes detectados:
['CO', 'NO', 'NO2', 'O3', 'PM10', 'PM2.5', 'SO2']

Rango de fechas:
2020-01-01 00:00:00 -> 2023-12-31 23:00:00


,provincia,municipio,estacion,magnitud,contaminante,punto_muestreo,archivo_origen,anno,mes,dia,hora,valor,fecha,datetime,anio,mes_num,mes_nombre,dia_mes,dia_semana_num,dia_semana,fin_de_semana,estacion_anio,franja_horaria
0,50,297,29,10,PM10,50297029_10_49,aire_2020.csv,2020,2,6,1,19.48,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada
1,50,297,29,14,O3,50297029_14_6,aire_2020.csv,2020,2,6,1,14.61,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada
2,50,297,32,1,SO2,50297032_1_38,aire_2020.csv,2020,2,6,1,1.77,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada
3,50,297,32,6,CO,50297032_6_48,aire_2020.csv,2020,2,6,1,0.22,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada
4,50,297,32,8,NO2,50297032_8_8,aire_2020.csv,2020,2,6,1,33.43,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada



Archivos guardados correctamente:
- ../data/processed/aire_total_unido.csv
- ../data/processed/aire_zaragoza_ancho.csv
- ../data/processed/aire_zaragoza_long.csv


In [23]:
df_check = pd.read_csv("../data/processed/aire_zaragoza_long.csv", parse_dates=["fecha", "datetime"])

print("Shape:", df_check.shape)
print("Columnas:", df_check.columns.tolist())

print("\nRango temporal:")
print(df_check["datetime"].min(), "->", df_check["datetime"].max())

print("\nContaminantes:")
print(sorted(df_check["contaminante"].dropna().unique().tolist()))

print("\nEstaciones:")
print(sorted(df_check["estacion"].dropna().unique().tolist()))

print("\nNulos:")
print(df_check.isnull().sum().sort_values(ascending=False).head(20))

print("\nDuplicados exactos:", df_check.duplicated().sum())

display(df_check.head())

Shape: (1304856, 23)
Columnas: ['provincia', 'municipio', 'estacion', 'magnitud', 'contaminante', 'punto_muestreo', 'archivo_origen', 'anno', 'mes', 'dia', 'hora', 'valor', 'fecha', 'datetime', 'anio', 'mes_num', 'mes_nombre', 'dia_mes', 'dia_semana_num', 'dia_semana', 'fin_de_semana', 'estacion_anio', 'franja_horaria']

Rango temporal:
2020-01-01 -> 2023-12-31 23:00:00

Contaminantes:
['CO', 'NO', 'NO2', 'O3', 'PM10', 'PM2.5', 'SO2']

Estaciones:
[26, 29, 32, 36, 37, 38, 39, 40]

Nulos:
valor             44840
municipio             0
provincia             0
magnitud              0
contaminante          0
punto_muestreo        0
estacion              0
archivo_origen        0
anno                  0
dia                   0
mes                   0
hora                  0
fecha                 0
datetime              0
anio                  0
mes_num               0
mes_nombre            0
dia_mes               0
dia_semana_num        0
dia_semana            0
dtype: int64

Duplicados ex

,provincia,municipio,estacion,magnitud,contaminante,punto_muestreo,archivo_origen,anno,mes,dia,hora,valor,fecha,datetime,anio,mes_num,mes_nombre,dia_mes,dia_semana_num,dia_semana,fin_de_semana,estacion_anio,franja_horaria
0,50,297,29,10,PM10,50297029_10_49,aire_2020.csv,2020,2,6,1,19.48,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada
1,50,297,29,14,O3,50297029_14_6,aire_2020.csv,2020,2,6,1,14.61,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada
2,50,297,32,1,SO2,50297032_1_38,aire_2020.csv,2020,2,6,1,1.77,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada
3,50,297,32,6,CO,50297032_6_48,aire_2020.csv,2020,2,6,1,0.22,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada
4,50,297,32,8,NO2,50297032_8_8,aire_2020.csv,2020,2,6,1,33.43,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada


In [24]:
resumen_aire = (
    df_check.groupby("contaminante")
    .agg(
        registros=("valor", "size"),
        nulos=("valor", lambda x: x.isnull().sum()),
        media=("valor", "mean"),
        mediana=("valor", "median"),
        minimo=("valor", "min"),
        maximo=("valor", "max")
    )
    .sort_values("registros", ascending=False)
)

display(resumen_aire)

,registros,nulos,media,mediana,minimo,maximo
contaminante,,,,,,
NO2,279792,9423,20.023261,15.86,1.00,177.73
O3,279240,6014,50.935175,53.47,0.10,173.12
CO,244968,7262,0.198016,0.18,0.02,2.19
SO2,209904,3441,4.306574,3.99,0.03,42.48
PM10,200904,14618,18.194829,15.15,0.00,168.30
NO,69360,3009,7.760430,4.68,0.00,327.71
PM2.5,20688,1073,8.571669,7.45,0.10,93.69


In [29]:
import pandas as pd
import numpy as np
import requests
import os
from dotenv import load_dotenv
import os

# =========================================================
# CONFIGURACIÓN
# =========================================================



load_dotenv()

API_KEY = os.getenv("AEMET_API_KEY")
print("Clave cargada:", API_KEY[:10] + "..." if API_KEY else "No encontrada")
RUTA_PROCESSED = "../data/processed"
os.makedirs(RUTA_PROCESSED, exist_ok=True)

# Usa primero Aeropuerto. Si falla, luego probamos Valdespartera.
IDEMA = "9434"

# =========================================================
# FUNCIONES
# =========================================================

def normalizar_columnas(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(".", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
    )
    return df

def limpiar_numerica_esp(series):
    return pd.to_numeric(
        series.astype(str)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan, "Ip": "0", "ip": "0"})
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

def aemet_get(url, api_key):
    r = requests.get(url, params={"api_key": api_key}, timeout=60)
    r.raise_for_status()
    meta = r.json()

    if not isinstance(meta, dict) or "datos" not in meta:
        raise ValueError(f"Respuesta inesperada AEMET: {meta}")

    r2 = requests.get(meta["datos"], timeout=60)
    r2.raise_for_status()
    return r2.json()

def descargar_aemet_semestres(idema, api_key):
    periodos = [
        ("2020-01-01T00:00:00UTC", "2020-06-30T23:59:59UTC"),
        ("2020-07-01T00:00:00UTC", "2020-12-31T23:59:59UTC"),
        ("2021-01-01T00:00:00UTC", "2021-06-30T23:59:59UTC"),
        ("2021-07-01T00:00:00UTC", "2021-12-31T23:59:59UTC"),
        ("2022-01-01T00:00:00UTC", "2022-06-30T23:59:59UTC"),
        ("2022-07-01T00:00:00UTC", "2022-12-31T23:59:59UTC"),
        ("2023-01-01T00:00:00UTC", "2023-06-30T23:59:59UTC"),
        ("2023-07-01T00:00:00UTC", "2023-12-31T23:59:59UTC"),
    ]

    lista = []

    for inicio, fin in periodos:
        url = (
            "https://opendata.aemet.es/opendata/api/"
            f"valores/climatologicos/diarios/datos/"
            f"fechaini/{inicio}/fechafin/{fin}/estacion/{idema}"
        )

        print(f"Descargando {inicio[:10]} -> {fin[:10]}")

        try:
            datos = aemet_get(url, api_key)

            if isinstance(datos, list) and len(datos) > 0:
                df_temp = pd.DataFrame(datos)
                df_temp["periodo_inicio"] = inicio[:10]
                df_temp["periodo_fin"] = fin[:10]
                lista.append(df_temp)
                print("OK", df_temp.shape)
            else:
                print("Sin datos en ese tramo")

        except Exception as e:
            print(f"Error en tramo {inicio[:10]} -> {fin[:10]}: {e}")

    if len(lista) == 0:
        raise ValueError("No se han descargado datos meteorológicos.")

    df = pd.concat(lista, ignore_index=True)

    # Quitar duplicados por si AEMET repite algo entre tramos
    if "fecha" in df.columns:
        df = df.drop_duplicates(subset=["fecha"], keep="first").copy()
    else:
        df = df.drop_duplicates().copy()

    return df

# =========================================================
# 1) DESCARGAR METEOROLOGÍA
# =========================================================

df_meteo = descargar_aemet_semestres(IDEMA, API_KEY)
df_meteo = normalizar_columnas(df_meteo)

print("\nShape meteo bruto:", df_meteo.shape)
display(df_meteo.head())

# =========================================================
# 2) LIMPIAR METEOROLOGÍA
# =========================================================

df_meteo["fecha"] = pd.to_datetime(df_meteo["fecha"], errors="coerce")

if "prec" in df_meteo.columns:
    df_meteo["prec_traza"] = df_meteo["prec"].astype(str).str.lower().eq("ip")

columnas_numericas = [
    "altitud", "tmed", "prec", "tmin", "tmax",
    "velmedia", "racha", "sol", "presmax", "presmin"
]

for col in columnas_numericas:
    if col in df_meteo.columns:
        df_meteo[col] = limpiar_numerica_esp(df_meteo[col])

rename_map = {
    "indicativo": "idema",
    "nombre": "estacion_meteo",
    "provincia": "provincia_meteo",
    "altitud": "altitud_meteo",
    "tmed": "temp_media",
    "prec": "precipitacion",
    "tmin": "temp_min",
    "tmax": "temp_max",
    "velmedia": "viento_vel_media",
    "dir": "direccion_viento",
    "racha": "racha_max",
    "sol": "insolacion",
    "presmax": "presion_max",
    "presmin": "presion_min"
}

for old, new in rename_map.items():
    if old in df_meteo.columns:
        df_meteo.rename(columns={old: new}, inplace=True)

df_meteo["anio_meteo"] = df_meteo["fecha"].dt.year
df_meteo["mes_meteo"] = df_meteo["fecha"].dt.month
df_meteo["dia_meteo"] = df_meteo["fecha"].dt.day

columnas_utiles = [
    "fecha",
    "idema",
    "estacion_meteo",
    "provincia_meteo",
    "altitud_meteo",
    "temp_media",
    "precipitacion",
    "prec_traza",
    "temp_min",
    "temp_max",
    "viento_vel_media",
    "direccion_viento",
    "racha_max",
    "insolacion",
    "presion_max",
    "presion_min",
    "anio_meteo",
    "mes_meteo",
    "dia_meteo"
]

columnas_utiles = [c for c in columnas_utiles if c in df_meteo.columns]
df_meteo = df_meteo[columnas_utiles].copy()

print("\nShape meteo limpio:", df_meteo.shape)
print("\nNulos meteo:")
display(df_meteo.isnull().sum().sort_values(ascending=False))
display(df_meteo.head())

# =========================================================
# 3) CARGAR AIRE Y UNIR
# =========================================================

df_aire_zgz = pd.read_csv(
    f"{RUTA_PROCESSED}/aire_zaragoza_long.csv",
    parse_dates=["fecha", "datetime"]
)

df_final = df_aire_zgz.merge(
    df_meteo,
    on="fecha",
    how="left"
)

print("\nShape aire:", df_aire_zgz.shape)
print("Shape final:", df_final.shape)
print("\nNulos tras unión:")
display(df_final.isnull().sum().sort_values(ascending=False).head(25))
display(df_final.head())

# =========================================================
# 4) GUARDAR
# =========================================================

df_meteo.to_csv(f"{RUTA_PROCESSED}/aemet_zaragoza_diario.csv", index=False)
df_final.to_csv(f"{RUTA_PROCESSED}/dataset_final_aire_meteo_zaragoza.csv", index=False)

print("\nArchivos guardados:")
print(f"- {RUTA_PROCESSED}/aemet_zaragoza_diario.csv")
print(f"- {RUTA_PROCESSED}/dataset_final_aire_meteo_zaragoza.csv")

Clave cargada: eyJhbGciOi...
Descargando 2020-01-01 -> 2020-06-30
OK (182, 27)
Descargando 2020-07-01 -> 2020-12-31
OK (184, 27)
Descargando 2021-01-01 -> 2021-06-30
OK (181, 27)
Descargando 2021-07-01 -> 2021-12-31
OK (184, 27)
Descargando 2022-01-01 -> 2022-06-30
OK (181, 27)
Descargando 2022-07-01 -> 2022-12-31
OK (184, 27)
Descargando 2023-01-01 -> 2023-06-30
OK (181, 27)
Descargando 2023-07-01 -> 2023-12-31
Error en tramo 2023-07-01 -> 2023-12-31: 500 Server Error: Internal Server Error for url: https://opendata.aemet.es/opendata/sh/3343e390

Shape meteo bruto: (1277, 27)


,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,horatmax,dir,velmedia,racha,horaracha,sol,presmax,horapresmax,presmin,horapresmin,hrmedia,hrmax,horahrmax,hrmin,horahrmin,periodo_inicio,periodo_fin
0,2020-01-01,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,"1,0","0,0","0,3",06:10,"1,8",14:50,28,"1,7","5,6",04:40,"0,0","1004,6",10,"1001,9",15,95,98,Varias,91,00:00,2020-01-01,2020-06-30
1,2020-01-02,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,"0,6","0,0","-0,3",08:10,"1,6",18:40,29,"1,1","3,6",00:40,"0,0","1003,4",10,"1000,7",16,95,99,07:10,92,18:40,2020-01-01,2020-06-30
2,2020-01-03,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,"4,2","0,0","-0,1",07:10,"8,5",19:10,30,"4,4","8,3",21:40,"2,4","1003,6",10,"1000,7",Varias,87,97,06:00,76,14:30,2020-01-01,2020-06-30
3,2020-01-04,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,"8,6","0,0","3,6",06:40,"13,6",14:50,33,"5,0","12,5",12:30,"8,2","1003,9",10,"1001,2",15,76,97,06:40,58,14:40,2020-01-01,2020-06-30
4,2020-01-05,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,"8,2","0,0","3,0",23:00,"13,3",15:20,30,"3,6","10,3",05:30,"8,9","1001,9",00,"996,9",Varias,67,99,23:50,46,15:00,2020-01-01,2020-06-30



Shape meteo limpio: (1277, 19)

Nulos meteo:


insolacion          17
presion_min          3
presion_max          2
direccion_viento     1
racha_max            1
fecha                0
idema                0
estacion_meteo       0
provincia_meteo      0
temp_media           0
altitud_meteo        0
viento_vel_media     0
temp_max             0
temp_min             0
prec_traza           0
precipitacion        0
anio_meteo           0
mes_meteo            0
dia_meteo            0
dtype: int64

,fecha,idema,estacion_meteo,provincia_meteo,altitud_meteo,temp_media,precipitacion,prec_traza,temp_min,temp_max,viento_vel_media,direccion_viento,racha_max,insolacion,presion_max,presion_min,anio_meteo,mes_meteo,dia_meteo
0,2020-01-01,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,1.0,0.0,False,0.3,1.8,1.7,28,5.6,0.0,1004.6,1001.9,2020,1,1
1,2020-01-02,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,0.6,0.0,False,-0.3,1.6,1.1,29,3.6,0.0,1003.4,1000.7,2020,1,2
2,2020-01-03,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,4.2,0.0,False,-0.1,8.5,4.4,30,8.3,2.4,1003.6,1000.7,2020,1,3
3,2020-01-04,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,8.6,0.0,False,3.6,13.6,5.0,33,12.5,8.2,1003.9,1001.2,2020,1,4
4,2020-01-05,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249,8.2,0.0,False,3.0,13.3,3.6,30,10.3,8.9,1001.9,996.9,2020,1,5



Shape aire: (1304856, 23)
Shape final: (1304856, 41)

Nulos tras unión:


insolacion          204024
presion_min         192408
presion_max         191592
racha_max           190752
direccion_viento    190752
idema               189912
anio_meteo          189912
estacion_meteo      189912
altitud_meteo       189912
mes_meteo           189912
dia_meteo           189912
provincia_meteo     189912
precipitacion       189912
prec_traza          189912
temp_min            189912
temp_max            189912
viento_vel_media    189912
temp_media          189912
valor                44840
provincia                0
punto_muestreo           0
contaminante             0
magnitud                 0
estacion                 0
municipio                0
dtype: int64

,provincia,municipio,estacion,magnitud,contaminante,punto_muestreo,archivo_origen,anno,mes,dia,hora,valor,fecha,datetime,anio,mes_num,mes_nombre,dia_mes,dia_semana_num,dia_semana,fin_de_semana,estacion_anio,franja_horaria,idema,estacion_meteo,provincia_meteo,altitud_meteo,temp_media,precipitacion,prec_traza,temp_min,temp_max,viento_vel_media,direccion_viento,racha_max,insolacion,presion_max,presion_min,anio_meteo,mes_meteo,dia_meteo
0,50,297,29,10,PM10,50297029_10_49,aire_2020.csv,2020,2,6,1,19.48,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249.0,7.6,0.0,False,1.0,14.1,3.1,13,8.3,9.7,997.0,992.6,2020.0,2.0,6.0
1,50,297,29,14,O3,50297029_14_6,aire_2020.csv,2020,2,6,1,14.61,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249.0,7.6,0.0,False,1.0,14.1,3.1,13,8.3,9.7,997.0,992.6,2020.0,2.0,6.0
2,50,297,32,1,SO2,50297032_1_38,aire_2020.csv,2020,2,6,1,1.77,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249.0,7.6,0.0,False,1.0,14.1,3.1,13,8.3,9.7,997.0,992.6,2020.0,2.0,6.0
3,50,297,32,6,CO,50297032_6_48,aire_2020.csv,2020,2,6,1,0.22,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249.0,7.6,0.0,False,1.0,14.1,3.1,13,8.3,9.7,997.0,992.6,2020.0,2.0,6.0
4,50,297,32,8,NO2,50297032_8_8,aire_2020.csv,2020,2,6,1,33.43,2020-02-06,2020-02-06,2020,2,Febrero,6,3,Jueves,False,Invierno,Madrugada,9434,"ZARAGOZA, AEROPUERTO",ZARAGOZA,249.0,7.6,0.0,False,1.0,14.1,3.1,13,8.3,9.7,997.0,992.6,2020.0,2.0,6.0



Archivos guardados:
- ../data/processed/aemet_zaragoza_diario.csv
- ../data/processed/dataset_final_aire_meteo_zaragoza.csv
